# Data Leakage Prevention (DLP) LoRA Fine-Tuning
**This notebook fine-tunes a lightweight model (Google Gemma-2b-it) using 4-bit quantization and LoRA to classify DLP actions.**

## Section 1 — Setup

Install dependencies and import libraries. Note: `google/gemma-2b-it` requires huggingface authentication. You might need to add a Kaggle Secret called `HF_TOKEN`.

In [ ]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate trl

In [ ]:
import os
import torch
import json
import shutil
from huggingface_hub import login
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from IPython.display import FileLink, display

# If using Gemma, log in with Hugging Face Token from Kaggle Secrets:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN"))

## Section 2 — Configuration
Define dataset path and output model path. `google/gemma-2b-it` easily fits within 4GB VRAM using 4-bit quantization. If using Kaggle T4 x2, `device_map='auto'` automatically utilizes available GPUs.

In [ ]:
# --- CONFIGURATION ---
# Note: Replace this with your actual Kaggle Input path when uploaded
DATASET_PATH = "/kaggle/input/dlp-dataset/dlp_ml_dataset.jsonl"  

OUTPUT_DIR = "/kaggle/working/dlp_lora_package"
MODEL_ID = "google/gemma-2b-it" # 2B Parameter model

# Ensure saving directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Section 3 — Load Dataset
Load the `.jsonl` file and convert it to a HuggingFace Dataset format (all 6000 samples for training).

In [ ]:
def load_jsonl(file_path):
    data = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line))
        print(f"Loaded {len(data)} samples from {file_path}")
    else:
        print(f"Warning: {file_path} not found. Generating dummy sample for testing notebook structure locally.")
        dummy_row = {
            "text": "Customer email is john@corp.com",
            "surface": "OUTPUT",
            "label": "REDACT",
            "features": {"num_emails": 1, "has_db_connection": False, "max_entropy": 3.2}
        }
        data = [dummy_row] * 60  # Generate enough dummy rows for tests if needed
    return data

dataset_list = load_jsonl(DATASET_PATH)
dataset = Dataset.from_list(dataset_list)

print("\nDataset length:", len(dataset))
print("Sample keys:", dataset[0].keys())

## Section 4 — Preprocessing
Convert each sample into the structured instruction prompt. The model will learn to output strictly the target label: `ALLOW`, `REDACT`, `ESCALATE`, or `BLOCK`.

In [ ]:
def format_dlp_prompt(example):
    # Extract features safely
    features = example.get("features", {})
    features_str = "\n".join([f"{k}={v}" for k, v in features.items()])
    
    # Construct the strictly formatted prompt matching the requirement
    prompt = f"""[SURFACE={example.get('surface', 'UNKNOWN')}]

[FEATURES]
{features_str}

[TEXT]
{example.get('text', '')}

Classify the risk level:
Target Output
{example.get('label', 'UNKNOWN')}"""
    
    return {"formatted_prompt": prompt}

# Apply formatting mapping across whole dataset (all 6000 items)
dataset = dataset.map(format_dlp_prompt)

print("\n--- Example Formatted Target Prompt ---\n")
print(dataset[0]["formatted_prompt"])

## Section 5 — Tokenization
Load the tokenizer and ensure padding and truncation are handled so sequence lengths are uniform. We use SFTTrainer which handles the tokenization step internally via `dataset_text_field`.

In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Gemma and Llama models often require this setup for padding to prevent sequence length issues
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" 

## Section 6 — Model Loading
Load the base model using 4-bit float quantization (`BitsAndBytesConfig`). Also enable gradient checkpointing to drastically save memory during backpropagation.

In [ ]:
# 4-bit Quantization Config (Great for T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # fp16 compute for Kaggle T4 GPUs
)

print(f"Loading base model ({MODEL_ID}) in 4-bit...")
# `device_map="auto"` works perfectly for T4x2
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False # Required for gradient checkpointing

# Prepare for custom k-bit training (Gradient Checkpointing activated internally)
model = prepare_model_for_kbit_training(model)

## Section 7 — Apply LoRA
Attach Low-Rank Adaptation (LoRA) adapters to the attention blocks. This freezes the base model and only updates new compact weights.

In [ ]:
# Configure LoRA Parameters
lora_config = LoraConfig(
    r=8,                  # Rank
    lora_alpha=32,        # Scale
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Target attention heads for Gemma/Llama
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach adapters to the model
model = get_peft_model(model, lora_config)

# Verify we only train ~0.1% parameters
model.print_trainable_parameters()

## Section 8 & 9 — Training Setup & Execution
Define hyperparameters suitable for a T4 setup. Small batch size + gradient accumulation steps = stable simulated large batch. Train for 1 epoch and max 300 steps as requested.

In [ ]:
# Configure Training Hyperparameters 
training_args = TrainingArguments(
    output_dir="./lora_checkpoints",
    per_device_train_batch_size=2,  # Using 2 perfectly fits a 16GB T4 GPU
    gradient_accumulation_steps=4,  # Effective batch size = 8
    max_steps=300,                  # Exactly 300 steps as per specification
    learning_rate=2e-4,             # Standard LoRA learning rate
    fp16=True,                      # Use fp16 for T4 capability
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    optim="paged_adamw_8bit",       # Memory efficient optimizer prevents OOM spikes
)

# Initialize Supervised Fine-Tuning Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config,
    dataset_text_field="formatted_prompt", # Automatically handles custom prompt text
    max_seq_length=512,             # Keep padded sequences heavily capped for performance
)

print("\nStarting LoRA Fine-Tuning...\n")
# Execute Training
trainer.train()

## Section 10 — Save Model
Save the fine-tuned LoRA adapters + Tokenizer. The output is tiny (approx 50-100MB).

In [ ]:
print(f"Applying final save to {OUTPUT_DIR}...")

# Save PEFT model adapters
trainer.model.save_pretrained(OUTPUT_DIR)

# Save the tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Model adapters and tokenizer successfully saved.")

## Section 11 — Export / Download
Zip the output folder containing the adapters, and generate a Kaggle direct download link.

In [ ]:
# Create a Zip file of the exported directory
zip_path = "dlp_lora_package"
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"Archived model to {zip_path}.zip")

# Generate a Kaggle-compatible download link
display(FileLink(f'{zip_path}.zip'))